# Detection Viewer

Loads the submitted model and runs it on a chosen sample from the dataset,
showing the fused point cloud together with the predicted boxes (colored by
class) and the ground-truth boxes (green).


In [1]:
%load_ext autoreload
%autoreload 2

## Setup

In [2]:
import os
import sys
import glob

REPO_ROOT = os.getcwd() if os.path.isdir("Mixed-Signals-Dataset") else os.path.dirname(os.getcwd())  # notebook lives at the repo root

if os.path.isdir("Mixed-Signals-Dataset"):
    os.chdir("Mixed-Signals-Dataset")

sys.path.insert(0, REPO_ROOT)

from msig_compat.dataset_patch import apply_all_patches
apply_all_patches()

import numpy as np
import plotly.graph_objects as go
from torch.utils.data import DataLoader

import opencood.hypes_yaml.yaml_utils as yaml_utils
from opencood.data_utils.datasets import build_dataset
from opencood.tools import train_utils, inference_utils

print("cwd:", os.getcwd())


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


cwd: /home/nitin/Nitin/LaneDetection/v2x-cooperative-detection/Mixed-Signals-Dataset


## Load model and test set

In [3]:
hypes = yaml_utils.msig_load_yaml("opencood/hypes_yaml/msig_pillar_attnfuse_local.yaml")
dataset = build_dataset(hypes, visualize=True, train=False)
print(f"{len(dataset)} samples available")

model = train_utils.create_model(hypes)
train_utils.openpcdet_load_weights(model, "opencood/logs/final_model/net_epoch50.pth", strict=True)
model.eval()
model.cuda()
print("model loaded")


MSigIntermediateFusion | load gt infos
MSigIntermediateFusion | building pre- & post-processor for training:=False
WARNING | @VoxelPostprocessorMulti, feature_stride is set to the default value of 2
1791 samples available
INFO | @ att_bev_backbone, self.use_freezed_rsu_downsampled_feat_maps: False
loaded weights for 142 / 142


model loaded


/home/nitin/Nitin/LaneDetection/v2x-cooperative-detection/Mixed-Signals-Dataset/opencood/utils/freezed_rsu_fusion_utils.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

## Plotting helpers

In [4]:
ptc_layout_config = {
    'paper_bgcolor': 'rgb(255,255,255)',
    'width': 800,
    'height': 800,
    'margin': {'l': 20, 'r': 20, 'b': 20, 't': 40},
    'showlegend': False,
    'scene': {
        'aspectmode': 'manual',
        'aspectratio': {'x': 0.75, 'y': 0.75, 'z': 0.05},
        'camera': {'eye': {'x': -0.1, 'y': 0, 'z': 0.15}},
        'xaxis': {'color': 'rgb(150,150,150)', 'dtick': 10, 'gridcolor': 'rgb(100,100,100)',
                  'range': [-60, 60], 'showbackground': False, 'tickprefix': 'x:'},
        'yaxis': {'color': 'rgb(150,150,150)', 'dtick': 10, 'gridcolor': 'rgb(100,100,100)',
                  'range': [-60, 60], 'showbackground': False, 'tickprefix': 'y:'},
        'zaxis': {'color': 'rgb(150,150,150)', 'dtick': 10, 'gridcolor': 'rgb(100,100,100)',
                  'range': [-10, 10], 'showbackground': False, 'tickprefix': 'z:'},
    },
}

CLASS_NAMES = {1: 'car', 2: 'bike', 3: 'pedestrian'}
PRED_COLORS = {1: 'blue', 2: 'orange', 3: 'red'}
GT_COLOR = 'green'


def rotz(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def cart2hom(pts):
    return np.hstack((pts, np.ones((pts.shape[0], 1), dtype=np.float32)))


def transform_points(pts, trans_mat):
    return (cart2hom(pts) @ trans_mat.T).reshape(-1, 4)[:, :3]


def compute_box(trans_mat, shape):
    w, l, h = shape
    x_corners = [l/2, l/2, -l/2, -l/2, l/2, l/2, -l/2, -l/2]
    y_corners = [w/2, -w/2, -w/2, w/2, w/2, -w/2, -w/2, w/2]
    z_corners = [-h/2, -h/2, -h/2, -h/2, h/2, h/2, h/2, h/2]
    corners_3d = np.vstack([x_corners, y_corners, z_corners]).T
    return transform_points(corners_3d, trans_mat)


def get_linemarks(trans_mat, shape):
    corners = compute_box(trans_mat, shape)
    mid_front = (corners[0] + corners[1]) / 2
    mid_left = (corners[0] + corners[3]) / 2
    mid_right = (corners[1] + corners[2]) / 2
    corners = np.vstack((corners, np.vstack([mid_front, mid_left, mid_right])))
    idx = [0, 8, 9, 10, 8, 1, 2, 3, 0, 4, 5, 1, 5, 6, 2, 6, 7, 3, 7, 4]
    return corners[idx, :]


def get_bbox_trace(box, name, color, width=3):
    x, y, z, dx, dy, dz, heading = box
    trans_mat = np.eye(4, dtype=np.float32)
    trans_mat[:3, 3] = (x, y, z)
    trans_mat[:3, :3] = rotz(heading)
    markers = get_linemarks(trans_mat, (dy, dx, dz))
    return go.Scatter3d(mode='lines', x=markers[:, 0], y=markers[:, 1], z=markers[:, 2],
                        line=dict(color=color, width=width), name=name)


## Visualize a sample

In [5]:
def resolve_sample_paths(sample_idx):
    """sample_idx -> (sequence id, frame index, ego point cloud file path).

    sample_idx just indexes into the held-out test set (0..len(dataset)-1);
    this resolves it back to the actual sequence/frame/file it corresponds
    to on disk, and is how you'd point this at a specific piece of data
    instead of a blind index.
    """
    seq_idx, idx_in_seq = dataset._convert_idx_in_dataset_to_idx_in_sequence(sample_idx)
    idx_in_seq = dataset.check_idx_in_seq_has_gt_boxes(seq_idx, idx_in_seq)
    matches = glob.glob(f"../data/mixed-signals/PointClouds/mini_{seq_idx}/top_{idx_in_seq}_*.pcd")
    return seq_idx, idx_in_seq, (matches[0] if matches else None)


def visualize_sample(sample_idx, score_threshold=0.3, num_points_shown=15000):
    seq_idx, idx_in_seq, pcd_path = resolve_sample_paths(sample_idx)
    print(f"sample {sample_idx} -> sequence mini_{seq_idx}, frame {idx_in_seq}, file: {pcd_path}")

    batch = dataset.collate_batch_test([dataset[sample_idx]])
    batch = train_utils.to_device(batch, device='cuda')

    pred_box_tensor, pred_score, pred_class, gt_box_tensor, gt_class = \
        inference_utils.inference_intermediate_fusion(batch, model, dataset, bandwidth_dict=None)

    pred_box = pred_box_tensor.cpu().numpy()
    pred_score = pred_score.cpu().numpy()
    pred_class = pred_class.cpu().numpy().astype(int)
    gt_box = gt_box_tensor.cpu().numpy()
    gt_class = gt_class.cpu().numpy().astype(int)

    keep = pred_score > score_threshold
    pred_box, pred_class, pred_score = pred_box[keep], pred_class[keep], pred_score[keep]

    counts = {name: int((pred_class == cid).sum()) for cid, name in CLASS_NAMES.items()}
    gt_counts = {name: int((gt_class == cid).sum()) for cid, name in CLASS_NAMES.items()}
    print(f"sample {sample_idx} - predictions: {counts}  |  ground truth: {gt_counts}")
    print("colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green")

    points = batch['metadata']['points'].cpu().numpy()
    if points.shape[0] > num_points_shown:
        points = points[np.random.choice(points.shape[0], num_points_shown, replace=False)]

    point_trace = go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='markers', marker=dict(size=0.8, color=points[:, -1], colorscale='Jet'))

    pred_traces = [get_bbox_trace(box, f'pred_{CLASS_NAMES.get(cls, cls)}_{i}', PRED_COLORS.get(cls, 'blue'))
                   for i, (box, cls) in enumerate(zip(pred_box, pred_class))]
    gt_traces = [get_bbox_trace(box, f'gt_{i}', GT_COLOR) for i, box in enumerate(gt_box)]

    fig = go.Figure(data=[point_trace] + pred_traces + gt_traces, layout=ptc_layout_config)
    fig.update_layout(title=f'sample {sample_idx}')
    fig.show()


## Sample views

A handful of samples spread across the held-out test sequences, run through
the model as-is.


In [6]:
visualize_sample(0)

sample 0 -> sequence mini_31, frame 2, file: ../data/mixed-signals/PointClouds/mini_31/top_2_1712122476.831163639.pcd


sample 0 - predictions: {'car': 6, 'bike': 3, 'pedestrian': 7}  |  ground truth: {'car': 6, 'bike': 6, 'pedestrian': 8}
colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green


/home/nitin/Nitin/LaneDetection/v2x-cooperative-detection/Mixed-Signals-Dataset/opencood/utils/box_utils.py:662: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  iou = torch.tensor(


In [7]:
visualize_sample(300)

sample 300 -> sequence mini_32, frame 1, file: ../data/mixed-signals/PointClouds/mini_32/top_1_1712122527.249335860.pcd


sample 300 - predictions: {'car': 5, 'bike': 4, 'pedestrian': 3}  |  ground truth: {'car': 4, 'bike': 3, 'pedestrian': 5}
colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green


/home/nitin/Nitin/LaneDetection/v2x-cooperative-detection/Mixed-Signals-Dataset/opencood/utils/box_utils.py:662: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  iou = torch.tensor(


In [8]:
visualize_sample(600)

sample 600 -> sequence mini_33, frame 3, file: ../data/mixed-signals/PointClouds/mini_33/top_3_1712122580.577496137.pcd


sample 600 - predictions: {'car': 10, 'bike': 3, 'pedestrian': 3}  |  ground truth: {'car': 9, 'bike': 4, 'pedestrian': 4}
colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green


In [9]:
visualize_sample(900)

sample 900 -> sequence mini_34, frame 5, file: ../data/mixed-signals/PointClouds/mini_34/top_5_1712122653.711684349.pcd


sample 900 - predictions: {'car': 4, 'bike': 4, 'pedestrian': 4}  |  ground truth: {'car': 4, 'bike': 4, 'pedestrian': 7}
colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green


In [10]:
visualize_sample(1200)

sample 1200 -> sequence mini_35, frame 6, file: ../data/mixed-signals/PointClouds/mini_35/top_6_1712122693.236153061.pcd


sample 1200 - predictions: {'car': 7, 'bike': 3, 'pedestrian': 3}  |  ground truth: {'car': 9, 'bike': 3, 'pedestrian': 6}
colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green


## Try your own

Pick any index in `[0, len(dataset))` and re-run.

In [11]:
sample_idx = 4  # edit this
visualize_sample(sample_idx)


sample 4 -> sequence mini_31, frame 4, file: ../data/mixed-signals/PointClouds/mini_31/top_4_1712122477.28956044.pcd


sample 4 - predictions: {'car': 6, 'bike': 3, 'pedestrian': 6}  |  ground truth: {'car': 6, 'bike': 6, 'pedestrian': 8}
colors - predictions: car=blue, bike=orange, pedestrian=red | ground truth: green
